##### current_date()
- is used to return **current system date** `without time` in PySpark **DateType**.
- The **date** will be in the format **"yyyy-MM-dd"**.

It is very commonly used in:
- `ETL load date columns`
- `Audit columns`
- `Partitioning`
- `Filtering recent data`
- `Incremental loads`
- `Watermarking`
- `Data retention logic`

**Timezone Awareness**
- `If timezone changes → date may change`.
- current_date() depends on:

      spark.conf.get("spark.sql.session.timeZone")

##### Syntax

     current_date()

- `The braces are optional`.

**Arguments:**
- `This function takes no arguments`.

**Returns:**
- `A DATE`.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import current_date, current_timestamp

##### 1) Basic Example

In [0]:
%sql
SELECT current_date() -- with braces
UNION ALL
SELECT current_date;  -- without braces

current_date()
2026-02-24
2026-02-24


In [0]:
df = spark.range(1) \
  .select(current_date().alias("today")).display()

today
2026-02-24


In [0]:
# data = [("1",)] -- method 01
# method 02
data = [["1"]].   
df1 = spark.createDataFrame(data, ["id"])

# current_date() & current_timestamp()
df1.withColumn("current_date", current_date()) \
   .withColumn("current_timestamp", current_timestamp()) \
   .display()

id,current_date,current_timestamp
1,2026-02-24,2026-02-24T03:07:45.072Z


##### 2) Add as Audit Column

In [0]:
column_names = ["language", "framework", "users"]

data = [
    ("Python", "Django", 20000), 
    ("Python", "FastAPI", 9000), 
    ("Java", "Spring", 7000), 
    ("JavaScript", "ReactJS", 5000)
]
df_audit = spark.createDataFrame(data, column_names) \
    .withColumn("curr_date", current_date()) \
    .withColumn("curr_timestamp", current_timestamp())

display(df_audit)

language,framework,users,curr_date,curr_timestamp
Python,Django,20000,2026-02-26,2026-02-26T18:22:43.797Z
Python,FastAPI,9000,2026-02-26,2026-02-26T18:22:43.797Z
Java,Spring,7000,2026-02-26,2026-02-26T18:22:43.797Z
JavaScript,ReactJS,5000,2026-02-26,2026-02-26T18:22:43.797Z


In [0]:
df_cd = spark.read.csv("/Volumes/azureadb/pyspark/timestamp/current_date()_current_user()_current_timestamp().csv", header=True, inferSchema=True)
display(df_cd.limit(10))

Commodity_Index,Sensex_Category,Label_Type,Effective_Date,Income_Level,Department,TARGET,Input_Timestamp_UTC
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1709109264
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1710234895
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1709109264
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1707813327
DISCOUNT,Top,average,6-Mar-23,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Mar-23,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327


In [0]:
df_cd_audit = df_cd.withColumn("load_date", current_date())
display(df_cd_audit.limit(20))

Commodity_Index,Sensex_Category,Label_Type,Effective_Date,Income_Level,Department,TARGET,Input_Timestamp_UTC,load_date
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1709109264,2026-02-24
DISCOUNT,Top,average,6-Feb-23,Low,TESTING,SQL Server,1710234895,2026-02-24
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1709109264,2026-02-24
DISCOUNT,Top,average,8-Jan-24,Low,TESTING,SQL Server,1707813327,2026-02-24
DISCOUNT,Top,average,6-Mar-23,Low,TESTING,SQL Server,1707813327,2026-02-24
DISCOUNT,Forward,medium,6-Mar-23,Low,TESTING,SQL Server,1707813327,2026-02-24
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327,2026-02-24
DISCOUNT,Forward,medium,6-Jan-25,Low,TESTING,SQL Server,1707813327,2026-02-24
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327,2026-02-24
DISCOUNT,Forward,medium,6-Apr-23,Low,TESTING,SQL Server,1707813327,2026-02-24


In [0]:
df_cd_audit.createOrReplaceTempView("df_cd_audit")

In [0]:
%sql
SELECT load_date, COUNT(*) FROM df_cd_audit
GROUP BY load_date;

load_date,COUNT(*)
2026-02-24,110


##### 3) Filter Last 7 Days Data

In [0]:
from pyspark.sql.functions import date_sub, col

data = [
    (1, "A", "2026-02-20",),
    (2, "B", "2026-02-15",),
    (3, "C", "2026-02-01",),
    (4, "D", "2026-02-21",),
    (5, "E", "2026-02-22",),
    (6, "F", "2025-12-30",),
]

df_fltr = spark.createDataFrame(data, ["id", "name", "load_date"]) \
           .withColumn("load_date", col("load_date").cast("date"))
display(df_fltr)

id,name,load_date
1,A,2026-02-20
2,B,2026-02-15
3,C,2026-02-01
4,D,2026-02-21
5,E,2026-02-22
6,F,2025-12-30


In [0]:
df_fltr.filter(
    col("load_date") >= date_sub(current_date(), 7)
).display()

id,name,load_date
1,A,2026-02-20
4,D,2026-02-21
5,E,2026-02-22


##### 4) Partitioning Example

In [0]:
%sql
DROP TABLE IF EXISTS source_table;

CREATE TABLE source_table (
    id INT,
    product_name STRING,
    quantity INT,
    price DOUBLE
)
USING DELTA;

INSERT INTO source_table VALUES
(1, 'Laptop', 2, 55000.50),
(2, 'Mouse', 5, 499.99),
(3, 'Keyboard', 3, 1499.75),
(4, 'Monitor', 1, 12000.00);

SELECT * FROM source_table;

id,product_name,quantity,price
1,Laptop,2,55000.5
2,Mouse,5,499.99
3,Keyboard,3,1499.75
4,Monitor,1,12000.0


In [0]:
df_prt = spark.table("source_table")
display(df_prt)

id,product_name,quantity,price
1,Laptop,2,55000.5
2,Mouse,5,499.99
3,Keyboard,3,1499.75
4,Monitor,1,12000.0


In [0]:
df_with_date = df_prt.withColumn("load_date", current_date())
display(df_with_date)

id,product_name,quantity,price,load_date
1,Laptop,2,55000.5,2026-02-26
2,Mouse,5,499.99,2026-02-26
3,Keyboard,3,1499.75,2026-02-26
4,Monitor,1,12000.0,2026-02-26


In [0]:
df_with_date.write \
    .mode("overwrite") \
    .partitionBy("load_date") \
    .saveAsTable("parts_table")

In [0]:
%sql
SELECT * FROM parts_table;

id,product_name,quantity,price,load_date
1,Laptop,2,55000.5,2026-02-26
2,Mouse,5,499.99,2026-02-26
3,Keyboard,3,1499.75,2026-02-26
4,Monitor,1,12000.0,2026-02-26


| Table        | Contains load_date? | Partitioned? |
| ------------ | ------------------- | ------------ |
| source_table | ❌ No                | ❌ No         |
| parts_table  | ✅ Yes               | ✅ Yes        |

**How to Show Partitions?**

In [0]:
spark.sql("SHOW PARTITIONS parts_table").display()

load_date
2026-02-26


In [0]:
spark.sql("DESCRIBE FORMATTED parts_table").display()

col_name,data_type,comment
id,int,null
product_name,string,null
quantity,int,null
price,double,null
load_date,date,null
# Partition Information,,
# col_name,data_type,comment
load_date,date,null
,,
# Delta Statistics Columns,,


In [0]:
display(spark.read.table("workspace.default.parts_table"))

id,product_name,quantity,price,load_date
1,Laptop,2,55000.5,2026-02-26
2,Mouse,5,499.99,2026-02-26
3,Keyboard,3,1499.75,2026-02-26
4,Monitor,1,12000.0,2026-02-26


##### 5) Using SQL

In [0]:
spark.sql("SELECT current_date() AS today").display()

today
2026-02-23
